In [10]:
%pip install langchain langchain-community openai feedparser

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain-openai

  Using cached langchain_openai-1.1.14-py3-none-any.whl.metadata (3.1 kB)
  Using cached tiktoken-0.12.0-cp313-cp313-win_amd64.whl.metadata (6.9 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
Using cached langchain_openai-1.1.14-py3-none-any.whl (88 kB)
Using cached tiktoken-0.12.0-cp313-cp313-win_amd64.whl (879 kB)
Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl (277 kB)

   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [tiktoken]
   ------------- -------------------------- 1/3 [tiktoken]
   -------------------------- ------------- 2/3 [langchain-openai]
   -------------------------- ------------- 2/3 [langchain-openai]
   -------------------------- ------------- 2/3 [langchain-openai]
   -------------------------- ------------- 2/3 [langchain-openai]
   ---------------------------------------- 3/3 [langchain-openai]

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "sk-or-v1-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # OpenRouter key

llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
)

In [21]:
import feedparser
import requests

rss_url = "https://techcrunch.com/tag/artificial-intelligence/feed/"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

response = requests.get(rss_url, headers=headers, timeout=20)
response.raise_for_status()
feed = feedparser.parse(response.content)

articles = [entry.get("title", "").strip() for entry in feed.entries if entry.get("title")]
articles = articles[:10]

if not articles:
    raise ValueError("No headlines fetched from RSS feed. Try a different feed URL.")

print(f"Fetched {len(articles)} headlines")
articles

Fetched 10 headlines


['Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz',
 'Meta bought 1 GW of solar this week',
 'How one AI startup is helping rice farmers battle climate change',
 'Harvard dropouts to launch ‘always on’ AI smart glasses that listen and record every conversation',
 'Meta to add 100MW of solar power from US gear',
 'Perplexity accused of scraping websites that explicitly blocked AI scraping',
 'Obvio’s stop sign cameras use AI to root out unsafe drivers',
 'Breakneck data center growth challenges Microsoft’s sustainability goals',
 'Gridcare thinks more than 100 GW of data center capacity is hiding in the grid',
 'Meta adds another 650 MW of solar power to its AI push']

In [22]:
print(type(llm))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import sys

research_prompt = PromptTemplate.from_template(
    """
You are a research agent.

Analyze the following AI news headlines:
{articles}

Return:
- key topics
- short summaries
- important keywords

Output in structured format.
"""
)

formatted_articles = "\n".join(f"- {title}" for title in articles)
research_chain = research_prompt | llm
research_output = research_chain.invoke({"articles": formatted_articles})
print(research_output.content)

**AI‑Related News – Quick Analysis**

| # | Headline | Key Topics | Short Summary (≈1‑2 sentences) | Important Keywords |
|---|----------|------------|--------------------------------|--------------------|
| 1 | **Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz** | • AI‑driven biotech financing  <br>• Venture capital involvement  <br>• Cross‑industry talent (Meta, OpenAI, Wiz) | Converge Bio, a biotech startup that uses generative‑AI for protein/DNA design, closed a $25 million Series A round led by Bessemer Venture Partners with participation from former executives of Meta, OpenAI and Wiz. | Converge Bio, $25 M Series A, Bessemer, AI‑generated biotech, Meta execs, OpenAI alumni, Wiz |
| 2 | **Meta bought 1 GW of solar this week** | • Renewable energy procurement  <br>• AI data‑center power needs  <br>• Corporate sustainability | Meta announced it has purchased an additional 1 gigawatt of solar generation capacity to power its expanding AI and metaverse in

In [25]:
trend_prompt = PromptTemplate(
    input_variables=["data"],
    template="""
You are a trend detection agent.

Based on this research:
{data}

Identify:
- top trending topics
- why they are trending

Return only the top 3 topics.
"""
)
trend_chain = trend_prompt | llm
trends = trend_chain.invoke({"data": research_output})
print(trends.content)

**1. Renewable‑energy procurement for AI compute**  
*Why it’s trending:*  
- Meta alone announced three separate solar deals (1 GW, 100 MW, 650 MW) in a single week, pushing its clean‑energy portfolio past 2 GW to power ever‑larger AI and metaverse workloads.  
- The rapid build‑out of AI‑intensive data centres (e.g., Microsoft’s expansion) is outpacing the availability of renewable power, creating a high‑visibility tension between growth and net‑zero targets.  
- Utilities and analytics firms (Gridcare) are now spotlighting “hidden” data‑center demand, underscoring the industry‑wide scramble to secure carbon‑free electricity.

**2. AI moving into deep‑industry verticals**  
*Why it’s trending:*  
- Start‑ups are applying generative and computer‑vision AI to traditionally non‑tech sectors: Converge Bio (AI‑generated protein/DNA design), agritech for rice farming (precision water use, disease detection), Obvio’s AI‑enhanced stop‑sign cameras, and Gridcare’s AI grid‑analytics for hidden

In [27]:
selection_prompt = PromptTemplate(
    input_variables=["trends"],
    template="""
You are a content strategist.

From these topics:
{trends}

Select:
- ONLY 1 topic worth posting
- based on impact, novelty, and audience interest

Explain why.
"""
)

selection_chain = selection_prompt | llm
selected_topic = selection_chain.invoke({"trends": trends})
print(selected_topic.content)

**Chosen topic:** **Renewable‑energy procurement for AI compute**

### Why this topic wins on impact, novelty, and audience interest  

| Criterion | How the topic scores | Why it matters for your audience |
|-----------|----------------------|----------------------------------|
| **Impact** | • Directly ties the fastest‑growing tech (AI compute) to the biggest global climate priority (net‑zero). <br>• Involves megawatt‑scale contracts (1 GW + 100 MW + 650 MW) that reshape power‑grid planning and carbon‑emission trajectories. <br>• Sets a precedent: if Meta can lock in >2 GW of clean power, other AI‑heavy firms (Microsoft, Google, Amazon) will feel pressure to follow or face reputational risk. | Executives, investors, and policy‑makers need to understand the scale of the energy challenge now tied to AI, because decisions made today will dictate the carbon‑budget and cost structure of the next generation of data‑centers. |
| **Novelty** | • The “hidden‑load” narrative (Gridcare’s discov

In [29]:
content_prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
You are a Twitter content expert.

Create a viral Twitter post on:
{topic}

Structure:
- Strong hook
- Clear explanation
- Insight
- CTA

Make it engaging and human-like.
"""
)

content_chain = content_prompt | llm
post = content_chain.invoke({"topic": selected_topic})
print(post.content)

**🧵 1/ 🚀 AI is powering the future – but it’s also hungry for **clean** power. Meta just locked in **>2 GW** of renewable energy for its next‑gen compute. Here’s why this is a game‑changer for tech, investors, and the planet.**  

---  

**2/ 🌍 Impact**  
- 2 GW = enough electricity for ~1.5 M homes.  
- That’s the same load as a small city, now feeding AI models that power everything from chatbots to drug discovery.  
- When the world’s biggest AI player goes green, the pressure is on every other tech giant to follow or face a PR nightmare.  

---  

**3/ 🔎 The hidden‑load reveal**  
Gridcare’s latest analysis uncovered **“unseen” data‑center demand** that was skewing grid forecasts. Meta’s three‑deal sprint (1 GW + 100 MW + 650 MW) is the first public acknowledgment of that hidden load – and it’s reshaping how utilities plan for the next decade.  

---  

**4/ 💡 Novel business model**  
Long‑term PPAs + “green‑compute credits” = a new financial product.  
Investors can now bet on **c

In [31]:
hook_prompt = PromptTemplate(
    input_variables=["post"],
    template="""
Improve ONLY the first line (hook) of this post:

{post}

Make it more engaging and scroll-stopping.
"""
)

hook_chain = hook_prompt | llm
optimized_post = hook_chain.invoke({"post": post})
print(optimized_post.content)

**🧵 1/ 🚀 AI’s hunger for power just met its match – Meta is locking in **over 2 GW** of green electricity, enough to light up a small city and supercharge the next wave of intelligent machines.**


In [ ]:
%pip install streamlit

  Using cached streamlit-1.56.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached narwhals-2.19.0-py3-none-any.whl.me